# Hand Detection Dataset Cleaner

This notebook processes images in the dataset and moves only those with successfully detected hands to a clean directory structure.

## Import Libraries

In [1]:
import os
import shutil
import cv2 as cv
import logging
import mediapipe as mp
from tqdm import tqdm
from typing import List, Dict
import time

2025-05-16 12:40:51.886454: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-05-16 12:40:51.902662: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1747374051.922955   75158 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1747374051.928443   75158 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1747374051.942980   75158 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking 

## Configure Paths and Settings

In [2]:
# Input and output directories
INPUT_DIR = '/home/panha/Desktop/Rupp/Year_4/semester_2/thesis/hand_gesture_recognition_code/dataset/'
CLEAN_DIR = '/home/panha/Desktop/Rupp/Year_4/semester_2/thesis/hand_gesture_recognition_code/dataset_clean/'

# Hand gesture categories
CATEGORIES = ['nothing', 'up', 'turn_left', 'turn_right', 'down']

# MediaPipe settings
MIN_DETECTION_CONF = 0.7
STATIC_IMAGE_MODE = True
MAX_NUM_HAND = 1

print(f"Input directory: {INPUT_DIR}")
print(f"Clean output directory: {CLEAN_DIR}")

Input directory: /home/panha/Desktop/Rupp/Year_4/semester_2/thesis/hand_gesture_recognition_code/dataset/
Clean output directory: /home/panha/Desktop/Rupp/Year_4/semester_2/thesis/hand_gesture_recognition_code/dataset_clean/


## Configure Logging

In [3]:
logging.basicConfig(level=logging.INFO, format='%(levelname)s: %(message)s')
logger = logging.getLogger(__name__)

## Define Functions

In [4]:
def init_hands_model(confidence) -> mp.solutions.hands.Hands:
    """Initialize MediaPipe Hands model."""
    return mp.solutions.hands.Hands(
        static_image_mode=STATIC_IMAGE_MODE,
        max_num_hands=MAX_NUM_HAND,
        min_detection_confidence=confidence
    )

In [5]:
def get_image_paths_by_category(input_dir: str, categories: List[str]) -> Dict[str, List[str]]:
    """Get image paths organized by category."""
    valid_extensions = {'.jpg', '.jpeg', '.png', '.bmp', '.gif'}
    paths_by_category = {}
    
    for category in categories:
        category_dir = os.path.join(input_dir, category)
        if not os.path.exists(category_dir):
            logger.warning(f"Category directory not found: {category_dir}")
            continue
            
        paths_by_category[category] = []
        for file in os.listdir(category_dir):
            if os.path.splitext(file)[1].lower() in valid_extensions:
                paths_by_category[category].append(os.path.join(category_dir, file))
    
    return paths_by_category

In [10]:
def is_hand_detected(image_path: str, hands_model: mp.solutions.hands.Hands) -> bool:
    """Check if a hand is detected in the image."""
    try:
        image = cv.imread(image_path)
        if image is None:
            logger.warning(f"Failed to read image: {image_path}")
            return False
        
        # Convert image to RGB (MediaPipe requires RGB input)
        rgb_image = cv.cvtColor(image, cv.COLOR_BGR2RGB)
        
        # Process the image
        results = hands_model.process(rgb_image)
        
        # Return True if at least one hand is detected
        return results.multi_hand_landmarks is not None and len(results.multi_hand_landmarks) > 0
    except Exception as e:
        logger.warning(f"Error processing {image_path}: {str(e)}")
        return False

In [11]:
def setup_clean_directories(base_dir: str, categories: List[str]) -> None:
    """Create clean directory structure."""
    if not os.path.exists(base_dir):
        os.makedirs(base_dir)
        
    for category in categories:
        category_dir = os.path.join(base_dir, category)
        if not os.path.exists(category_dir):
            os.makedirs(category_dir)

## Initialize the Hands Model

In [12]:
hands = init_hands_model(MIN_DETECTION_CONF)
logger.info("MediaPipe Hands model initialized")

I0000 00:00:1747374169.344038   75158 gl_context_egl.cc:85] Successfully initialized EGL. Major : 1 Minor: 5
INFO: MediaPipe Hands model initialized
I0000 00:00:1747374169.346714   76049 gl_context.cc:369] GL version: 3.2 (OpenGL ES 3.2 Mesa 24.2.8-1ubuntu1~24.04.1), renderer: Mesa Intel(R) UHD Graphics (TGL GT1)


W0000 00:00:1747374169.379282   76042 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1747374169.408426   76040 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.


## Get Image Paths

In [13]:
image_paths_by_category = get_image_paths_by_category(INPUT_DIR, CATEGORIES)

# Display count of images in each category
for category, paths in image_paths_by_category.items():
    print(f"{category}: {len(paths)} images")

nothing: 3600 images
up: 1400 images
turn_left: 2800 images
turn_right: 2800 images
down: 2800 images


## Setup Clean Directory Structure

In [14]:
setup_clean_directories(CLEAN_DIR, CATEGORIES)
logger.info(f"Clean directory structure created at {CLEAN_DIR}")

INFO: Clean directory structure created at /home/panha/Desktop/Rupp/Year_4/semester_2/thesis/hand_gesture_recognition_code/dataset_clean/


## Process Images and Move Those with Detected Hands

In [15]:
start_time = time.time()
stats = {"detected": 0, "not_detected": 0}
category_stats = {category: {"detected": 0, "not_detected": 0} for category in CATEGORIES}

for category, image_paths in image_paths_by_category.items():
    logger.info(f"Processing {category} images...")
    
    progress_bar = tqdm(image_paths, desc=f"Processing {category}")
    
    for image_path in progress_bar:
        if is_hand_detected(image_path, hands):
            # Get the file name from the path
            file_name = os.path.basename(image_path)
            
            # Create the destination path
            dest_path = os.path.join(CLEAN_DIR, category, file_name)
            
            # Copy the image to the clean directory
            shutil.copy2(image_path, dest_path)
            
            stats["detected"] += 1
            category_stats[category]["detected"] += 1
        else:
            stats["not_detected"] += 1
            category_stats[category]["not_detected"] += 1

end_time = time.time()
processing_time = end_time - start_time

logger.info(f"Processing completed in {processing_time:.2f} seconds")
logger.info(f"Overall - Detected: {stats['detected']}, Not detected: {stats['not_detected']}")

for category in CATEGORIES:
    if category in category_stats:
        detected = category_stats[category]["detected"]
        not_detected = category_stats[category]["not_detected"]
        total = detected + not_detected
        if total > 0:
            detection_rate = (detected / total) * 100
            logger.info(f"{category} - Detected: {detected}, Not detected: {not_detected}, Detection rate: {detection_rate:.2f}%")

INFO: Processing nothing images...
Processing nothing: 100%|██████████| 3600/3600 [01:56<00:00, 30.85it/s]
INFO: Processing up images...
Processing up: 100%|██████████| 1400/1400 [00:45<00:00, 30.48it/s]
INFO: Processing turn_left images...
Processing turn_left: 100%|██████████| 2800/2800 [01:41<00:00, 27.70it/s]
INFO: Processing turn_right images...
Processing turn_right: 100%|██████████| 2800/2800 [01:30<00:00, 30.78it/s]
INFO: Processing down images...
Processing down: 100%|██████████| 2800/2800 [01:36<00:00, 28.87it/s]
INFO: Processing completed in 451.68 seconds
INFO: Overall - Detected: 12704, Not detected: 696
INFO: nothing - Detected: 3410, Not detected: 190, Detection rate: 94.72%
INFO: up - Detected: 1304, Not detected: 96, Detection rate: 93.14%
INFO: turn_left - Detected: 2747, Not detected: 53, Detection rate: 98.11%
INFO: turn_right - Detected: 2528, Not detected: 272, Detection rate: 90.29%
INFO: down - Detected: 2715, Not detected: 85, Detection rate: 96.96%


## Verify Clean Dataset

In [12]:
# Verify the clean dataset structure and counts
clean_stats = {}

for category in CATEGORIES:
    category_dir = os.path.join(CLEAN_DIR, category)
    if os.path.exists(category_dir):
        valid_extensions = {'.jpg', '.jpeg', '.png', '.bmp', '.gif'}
        clean_images = [f for f in os.listdir(category_dir) if os.path.splitext(f)[1].lower() in valid_extensions]
        clean_stats[category] = len(clean_images)

print("\nClean Dataset Statistics:")
for category, count in clean_stats.items():
    print(f"{category}: {count} images")

total_clean = sum(clean_stats.values())
print(f"\nTotal images in clean dataset: {total_clean}")


Clean Dataset Statistics:
nothing: 11725 images
up: 12043 images
turn_left: 13726 images
turn_right: 14596 images
down: 12123 images

Total images in clean dataset: 64213
